# Toploader Measurement

## Config
Edit the cell below for each new run. Standard toploader settings follow.

In [ ]:
# === CONFIG ===

# -- Per-run (edit these) --
# Alert recipients — registry names or raw IDs/addresses, single or list
ALERT_DISCORD = []  # e.g. ["asharpe"] or ["123456789012345678"]
ALERT_SLACK   = []  # e.g. ["asharpe"] or ["U0123456789"]
ALERT_EMAIL   = []  # e.g. ["asharpe"] or ["user@example.edu"]

# Gate sources: name -> {addr, limit, step}
GATES = {
    "k_tg": {"addr": "GPIB4::9::INSTR",  "limit": (-6.5, 6.5), "step": 0.005},
    "k_bg": {"addr": "GPIB4::10::INSTR", "limit": (-8.5, 7.0), "step": 0.005},
    "k_si": {"addr": "GPIB4::11::INSTR", "limit": (-50, 50),   "step": 0.005},
}

# -- Toploader standard (don't edit unless hardware changes) --
MAGNET_ADDR = "TCPIP0::192.168.0.101::7180::SOCKET"

SR830_ADDRS = {
    1: "GPIB0::8::INSTR",
    2: "GPIB0::9::INSTR",
    3: "GPIB0::4::INSTR",
    4: "GPIB0::1::INSTR",
    5: "GPIB1::5::INSTR",
    6: "GPIB0::6::INSTR",
    7: "GPIB0::7::INSTR",
    8: "TCPIP::192.168.0.106::inst0::INSTR",  # SR860
    9: "GPIB5::27::INSTR",
}

SR830_GAINS = {
    1: 100, 2: 100, 3: 100, 4: -1E6,
    5: 100, 6: 100, 7: 100, 8: 100, 9: 100,
}

KEITHLEY_DEFAULTS = {
    "mode": "VOLT",
    "rangev": 21,
    "rangei": 1e-6,
    "compliancev": 20,
    "compliancei": 1e-6,
}


# Rotator (optional — ANC350 two-axis stepper read back by lock-ins).
# enabled=True on the toploader (rotator present); set False for inserts without it.
ROTATOR = {
    "enabled": True,
    "dll_dir": r"C:\Users\toploader\Desktop\Software_ANC350v3\ANC350_DLL\Win_64Bit",
    # Defaults — per-axis entries below may override amplitude_mv / frequency_hz.
    "amplitude_mv": 60_000,        # 60 V piezo drive (param is mV, not V!)
    "frequency_hz": 100,           # 1 Hz idle; 100 Hz for sweeps
    "v_bounds": (-50e-3, 50e-3),   # absolute lock-in V clamp on servo targets
    # axis -> chassis axis idx, readback lock-in idx, calibrated direction_sign,
    #         (+ optional per-axis amplitude_mv / frequency_hz)
    "axes": {
        "theta": {"axis": 0, "readback_idx": 6, "direction_sign": +1},  # cookbook 2026-05-14
        "phi":   {"axis": 1, "readback_idx": 7, "direction_sign": -1},  # confirmed 2026-05
    },
}

---
## Setup
Run all cells below to initialize instruments.

In [ ]:
import numpy as np
import qcodes
from qcodes import validators
import time
import os
import sweep
from sweep import raster
import pyvisa as visa
from IPython.display import clear_output

from qcodes.instrument_drivers.stanford_research import SR830, SR860
from qcodes.instrument_drivers.Keithley import Keithley2400
from qcodes.instrument_drivers.american_magnetics import AMIModel430

from sharpelab_nb import Status
from sharpelab_nb.constants import TOPLOADER
from sharpelab_nb.thermometry import temperature_from_logfile, temperature_from_logfiles
from sharpelab_nb.drivers import SRS_DC205, Yokogawa7651
from sharpelab_nb.drivers.anc350 import ANC350
from sharpelab_nb.drivers.lockin_servo import bring_up_servo_axis, thermalized_setpoint

In [ ]:
# Alerts
from measurebot import alerts
alerts.show_config()
alerts.set_defaults(discord_user=ALERT_DISCORD, slack_user=ALERT_SLACK, email_user=ALERT_EMAIL)
alerts.slack(f"\U0001f40d connected \U0001f40d")

In [ ]:
# VISA
rm = visa.ResourceManager()
print(rm.list_resources())

In [ ]:
sweep.set_basedir('./data')

In [ ]:
# Magnet
mag = AMIModel430("magnet", address=MAGNET_ADDR)
print(f"Field: {mag.field()} T")

In [ ]:
# Lock-in amplifiers
sr830s_by_idx = {}
for idx, addr in SR830_ADDRS.items():
    name = f"sr830_{idx}"
    if "TCPIP" in addr:
        sr830s_by_idx[idx] = SR860(name, addr)
    else:
        sr830s_by_idx[idx] = SR830(name, addr)

sr830s = [(sr830s_by_idx[i], SR830_GAINS[i]) for i in sorted(sr830s_by_idx)]

for i, (srs, gain) in enumerate(sr830s):
    idx = sorted(sr830s_by_idx)[i]
    print(f"  {idx}: X={srs.X():.4e}  gain={gain}")

# Expose as individual variables for convenience
for idx, inst in sr830s_by_idx.items():
    globals()[f"sr830_{idx}"] = inst

In [ ]:
# Rotator servos (optional — see ROTATOR in CONFIG; Daisy.exe must be closed)
servos, thermalized = {}, {}
if ROTATOR["enabled"]:
    anc = ANC350("anc", dll_dir=ROTATOR["dll_dir"], num_axes=len(ROTATOR["axes"]))
    for nm, ax in ROTATOR["axes"].items():
        lockin = sr830s_by_idx[ax["readback_idx"]]
        servo = bring_up_servo_axis(
            f"servo_{nm}", anc.axes[ax["axis"]], lockin.X,
            direction_sign=ax["direction_sign"],
            amplitude_mv=ax.get("amplitude_mv", ROTATOR["amplitude_mv"]),
            frequency_hz=ax.get("frequency_hz", ROTATOR["frequency_hz"]),
            v_min=ROTATOR["v_bounds"][0], v_max=ROTATOR["v_bounds"][1],
        )
        servos[nm] = servo
        thermalized[nm] = thermalized_setpoint(servo, lockin)
        globals()[f"servo_{nm}"] = servo
        globals()[f"{nm}_thermalized"] = thermalized[nm]
        print(f"  {nm}: axis {ax['axis']}, sr830_{ax['readback_idx']}, "
              f"pos={servo.voltage()*1e3:+.4f} mV")

In [ ]:
# Gate sources
gate_instruments = {}
for name, cfg in GATES.items():
    k = Keithley2400(name, cfg["addr"])
    # k.mode.set(KEITHLEY_DEFAULTS["mode"])
    # k.rangev.set(KEITHLEY_DEFAULTS["rangev"])
    # k.rangei.set(KEITHLEY_DEFAULTS["rangei"])
    # k.compliancev.set(KEITHLEY_DEFAULTS["compliancev"])
    # k.compliancei.set(KEITHLEY_DEFAULTS["compliancei"])
    # k.output.set(1)
    k.volt.step = cfg["step"]
    lo, hi = cfg["limit"]
    k.volt.remove_validator()
    k.volt.add_validator(validators.Numbers(lo, hi))
    gate_instruments[name] = k
    globals()[name] = k
    print(f"  {name}: {cfg['addr']}  limits=({lo}, {hi})")

In [ ]:
# Temperature readout. The probe reads whichever logfile is live: two probes
# are racked (1=straight, 2=rotator) and only the installed one's file
# advances. temperature_from_logfiles tail-reads each candidate, uses the
# in-line timestamp to pick the freshest, and warns (non-fatally) if even that
# is >300 s stale — so a dead worker / wrong probe is auto-skipped, not read
# stale (zf1 2026-06-23: probe stuck 33.7 mK for 3.3 d).
temperature = temperature_from_logfiles("T", TOPLOADER.temp_logfile_candidates["T_probe"])
fridge_temperature = temperature_from_logfiles("T_fridge", TOPLOADER.temp_logfile_candidates["T_fridge"])
mag_temperature = temperature_from_logfiles("T_mag", TOPLOADER.temp_logfile_candidates["T_magnet"])

print(f"Probe:  {temperature()} K")
print(f"Fridge: {fridge_temperature()} K")
print(f"Magnet: {mag_temperature()} K")

In [ ]:
# UPS safe shutdown
from sharpelab_nb.ups import start_ups_monitor

def safe_shutdown():
    s = sweep.AsyncStation()
    s.ramp(k_bg.volt, 0)
    s.ramp(k_tg.volt, 0)
    s.ramp(k_si.volt, 0)
    s.ramp(mag.field, 0)
    if ROTATOR["enabled"]:
        for ax in anc.axes:
            ax.output_enabled(False)   # cut rotator HV; don't move it mid-event

ups_monitor = start_ups_monitor(safe_shutdown)

---
## Measurements
Define your measurement config before each sweep. The `mc` dict maps lock-ins
to contact pairs; `contact_pairs` assigns roles (longitudinal, hall, current)
to groups of channels. Update both as you rewire.

In [ ]:
from sharpelab_nb import build_measurement_config

# Channel -> contact wiring (update when you rewire)
mc = {
    "sr830_1": "29-28",
    "sr830_2": "28-25",
    "sr830_3": "25-24",
    "sr830_4": "I_{30-17}",
    "sr830_5": "13-14",
    "sr830_6": "10-11",
    "sr830_7": "11-13",
    "sr830_8": "14-15",
    "sr830_9": "24-22",
}

# Role -> which lock-in channels measure it
contact_pairs = {
    "long": {"channels": [1, 2, 3]},
    "hall": {"channels": [5, 6, 7]},
    "curr": {"channels": [4]},
}

measurement_config = build_measurement_config(mc, contact_pairs)

In [ ]:
# Status — customize with the parameters you care about
status = Status(
    field=mag.field,
    T=temperature,
    **{name: inst.volt for name, inst in gate_instruments.items()},
    **{nm: s.voltage for nm, s in servos.items()},
)
status()